# Comparison: ADD_TO_TRAIN × discard_criteria

For each combination of dataset filter (`discard_criteria`) and train injection (`ADD_TO_TRAIN`), runs the selected model(s) on all tasks and produces a histogram via `user_study_plot_hist`.

**Dataset configs (4):**
1. `all` — no filtering
2. `no_distractor` — discard distractor
3. `no_camera` — discard camera_not_working
4. `good_only` — discard all known bad criteria

**ADD_TO_TRAIN configs (3):**
- `[]` — no injection
- `[0,1,2]` — one trajectory per label moved from test to train
- `[0,1,2,0,1,2]` — two trajectories per label

Histograms are saved under `auto_fig_generator/add_to_train_comparison/`.

In [ ]:
import numpy as np
from tqdm import tqdm
from nocode_robot_programming.state_decision.utils import (
    kill_other_ipykernels, user_study_tasks_only, user_study_plot_hist_grouped, set_seed
)
from nocode_robot_programming.state_decision.state_decider import (
    StateDeciderBase, DINOFeaturePresence, DINOFeaturePresenceConcat,
    DINOFeaturePresenceAttnGated, DINOWithMIL, StateDeciderSIFT, AEGP
)
from nocode_robot_programming.state_decision_dataset_prepare.dataset_auto import TrajectoryDatasetEvaluationViewBuilder
from nocode_robot_programming.state_decision.plots.benchmark_plot import visualize_accuracies
from nocode_robot_programming.jupyter_plot import jupyter_plot as ipt
kill_other_ipykernels(force=True)
set_seed()

CRITERIA_CSV = "trajectory_criteria.csv"
TASKS = ['peg_pick', 'probe', 'wrap']
ANOMALY_DATASET = False
OUT_DIR = "add_to_train_comparison"

DECIDERS = [
    DINOFeaturePresenceAttnGated(dino_variant="dinov2_vits14", percentile_keep=None, attn_mode="hard", head_reduce="mean", attn_keep=0.4),
    # DINOFeaturePresence(dino_variant="dinov2_vits14"),
    DINOFeaturePresenceConcat(dino_variant="dinov2_vits14"),
    # DINOWithMIL(dino_variant="dinov2_vits14", att_hidden=128, dropout_p=0.1, lr=1e-4, weight_decay=1e-3, epochs=1000),
    # StateDeciderSIFT(method="SIFT"),
]

DATASET_CONFIGS = [
    {"name": "good",        "use_criteria": frozenset([""])},
    {"name": "distractor",  "use_criteria": frozenset(["distractor"])},
    {"name": "camera",      "use_criteria": frozenset(["camera_not_working"])},
    {"name": "others",      "use_criteria": frozenset(["hand_in_scene", "target_features_not_visible_or_too_small",
                                                        "teaching_multiple_things_at_once", "very_bad_train_data"])},
]

ADD_TO_TRAIN_CONFIGS = [
    ([], "no_inject"),
    ([0, 1, 2], "inject_1each"),
    ([0, 1, 2, 0, 1, 2], "inject_2each"),
]

# Which dataset config feeds visualize_accuracies heatmaps
VISUALIZE_DC_NAME = "good"

BRACKETS = []

for decider in DECIDERS:
    # Accumulate train/test lists for visualize_accuracies, keyed by add_name
    viz = {add_name: {"train": [], "test": [], "names": []} for _, add_name in ADD_TO_TRAIN_CONFIGS}

    for add_to_train, add_name in ADD_TO_TRAIN_CONFIGS:
        grouped_stats = {}
        for dc in DATASET_CONFIGS:
            dataset_builder = TrajectoryDatasetEvaluationViewBuilder(
                criteria_csv=CRITERIA_CSV,
                sync_criteria_csv=False,
                print_criteria_report=False,
                use_criteria=dc["use_criteria"],
            )
            stats = {}
            desc = f"{dc['name']} / {add_name} / {decider.short_name}"
            for name in tqdm(user_study_tasks_only(dataset_builder, TASKS), desc=desc):
                for d_train, d_test, d_text in dataset_builder.load_eval_from_task(
                        name, anomaly=ANOMALY_DATASET, add_to_train=add_to_train):
                    try:
                        decider.train(d_train.X, d_train.y_int, d_train.y_cls)
                    except AssertionError:
                        continue
                    y_pred_test = decider.predict_many(d_test.X)
                    test_acc = (np.array(d_test.y_names) == np.array(y_pred_test)).mean()
                    stats[d_text] = test_acc
                    if dc["name"] == VISUALIZE_DC_NAME:
                        y_pred_train = decider.predict_many(d_train.X)
                        train_acc = (np.array(d_train.y_names) == np.array(y_pred_train)).mean()
                        viz[add_name]["train"].append(train_acc * 100)
                        viz[add_name]["test"].append(test_acc * 100)
                        viz[add_name]["names"].append(d_text.split(",")[0])
            grouped_stats[dc["name"]] = stats

        user_study_plot_hist_grouped(
            grouped_stats,
            BRACKETS,
            f"{add_name}_{decider.short_name}.pdf",
            print_howmany_over_90=True,
            bins=21,
            folder=OUT_DIR,
        )

    # Three heatmaps in the same cell output — one per injection level
    for _, add_name in ADD_TO_TRAIN_CONFIGS:
        d = viz[add_name]
        if not d["names"]:
            continue
        visualize_accuracies(
            [d["train"]], [d["test"]],
            [f"{decider.short_name} ({add_name})"],
            d["names"],
            out_dir=f"{OUT_DIR}/{add_name}",
            jupyter_plot=True,
        )
    ipt.show()